In [1]:
pip install gradio PyPDF2

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ============================================
# Smart Study Hub - Complete Fixed Version
# ============================================

import gradio as gr
import PyPDF2
import os
import csv
import re
import random
from math import ceil
from datetime import datetime

# ---------- CSV file names ----------
USERS_FILE = "users.csv"
PROGRESS_FILE = "progress.csv"
RESUME_FILE = "resume.csv"


# ============================================
# PART 1: File Setup
# ============================================

def ensure_files():
    if not os.path.exists(USERS_FILE):
        with open(USERS_FILE, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["username", "password"])

    if not os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["timestamp", "username", "topic", "technique", "score"])

    if not os.path.exists(RESUME_FILE):
        with open(RESUME_FILE, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["timestamp", "username", "topic", "technique", "notes"])

ensure_files()


# ============================================
# PART 2: Authentication (Login / Signup)
# ============================================

def register_user(username, password):
    username = username.strip()
    if not username or not password:
        return False, "❌ Username and password are required."

    with open(USERS_FILE, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row["username"] == username:
                return False, "❌ Username already exists. Please log in."

    with open(USERS_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([username, password])

    return True, f"✅ Account created. Welcome, **{username}**!"


def authenticate_user(username, password):
    username = username.strip()
    if not username or not password:
        return False, "❌ Username and password are required."

    with open(USERS_FILE, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row["username"] == username and row["password"] == password:
                return True, f"✅ Logged in as **{username}**."

    return False, "❌ Incorrect username or password."


def login_handler(username, password, is_new):
    if is_new:
        ok, msg = register_user(username, password)
    else:
        ok, msg = authenticate_user(username, password)

    current_user = username.strip() if ok else ""
    return msg, current_user


# ============================================
# PART 3: Notes / PDF Processing
# ============================================

def extract_text_from_pdf(file):
    if file is None:
        return ""
    try:
        reader = PyPDF2.PdfReader(file.name)
        text = ""
        for page in reader.pages:
            text += page.extract_text() or ""
        return text
    except Exception:
        return ""


def combine_notes(pdf_file, manual_text):
    text = extract_text_from_pdf(pdf_file)
    if manual_text:
        text += "\n" + manual_text
    return text.strip()


def split_sentences(text):
    parts = re.split(r'[.!?]\s+', text)
    return [s.strip() for s in parts if len(s.strip().split()) > 4]


def extract_words(sentence):
    return re.findall(r"\b[A-Za-z]{4,}\b", sentence)


# ============================================
# PART 4: Resume + Progress
# ============================================

def save_resume(username, topic, technique, notes_text):
    username = username.strip()
    if not username or not topic or not technique:
        return
    truncated = notes_text[:5000]
    with open(RESUME_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            datetime.now().isoformat(),
            username,
            topic,
            technique,
            truncated
        ])


def load_last_resume(username):
    username = username.strip()
    if not username:
        return "❌ Please log in to resume.", "", "", ""

    last_row = None
    with open(RESUME_FILE, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row["username"] == username:
                last_row = row

    if not last_row:
        return f"ℹ️ No previous sessions found for **{username}**.", "", "", ""

    topic = last_row["topic"]
    technique = last_row["technique"]
    notes = last_row["notes"]
    msg = (
        f"### 🔁 Resume Last Session\n"
        f"- **User:** {username}\n"
        f"- **Topic:** {topic}\n"
        f"- **Technique:** {technique}\n\n"
        f"Use the **{technique}** tab with the same topic to continue.\n\n"
        f"#### Notes Preview\n{notes[:800]}{'...' if len(notes) > 800 else ''}"
    )
    return msg, topic, notes, technique


def save_progress(username, topic, technique, score_percent):
    username = username.strip()
    if not username:
        return
    with open(PROGRESS_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            datetime.now().isoformat(),
            username,
            topic,
            technique,
            f"{score_percent:.1f}"
        ])


def render_dashboard(username):
    username = username.strip()
    if not username:
        return "❌ Please log in to view your dashboard."

    entries = []
    with open(PROGRESS_FILE, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row["username"] == username:
                entries.append(row)

    if not entries:
        return f"ℹ️ No progress data found for **{username}**."

    # Aggregate
    summary = {}
    for row in entries:
        key = (row["topic"], row["technique"])
        score = float(row["score"])
        if key not in summary:
            summary[key] = {
                "count": 0,
                "total": 0.0,
                "best": 0.0,
                "last": row["timestamp"]
            }
        summary[key]["count"] += 1
        summary[key]["total"] += score
        summary[key]["best"] = max(summary[key]["best"], score)
        summary[key]["last"] = row["timestamp"]

    md = f"### 📊 Progress Dashboard for **{username}**\n\n"
    md += "| Topic | Technique | Attempts | Avg Score (%) | Best Score (%) | Last Attempt |\n"
    md += "|-------|-----------|----------|----------------|----------------|--------------|\n"

    for (topic, tech), data in summary.items():
        avg = data["total"] / data["count"]
        md += f"| {topic} | {tech} | {data['count']} | {avg:.1f} | {data['best']:.1f} | {data['last'][:19]} |\n"

    return md


# ============================================
# PART 5: Concept Extraction, Flashcards & Quiz
# ============================================

def extract_concept_definitions(notes_text):
    """
    Extract (concept -> definition) pairs using multiple patterns:
    - 'X is ...'
    - 'X refers to ...'
    - 'X is defined as ...'
    - 'Concept: ...'
    - 'Concept - ...'
    """
    concepts = {}
    sentences = split_sentences(notes_text)

    patterns = [
        r"(\b[A-Za-z][A-Za-z0-9_ ]{2,50}\b)\s+is\s+(.*)",
        r"(\b[A-Za-z][A-Za-z0-9_ ]{2,50}\b)\s+refers to\s+(.*)",
        r"(\b[A-Za-z][A-Za-z0-9_ ]{2,50}\b)\s+is defined as\s+(.*)",
    ]

    for s in sentences:
        for p in patterns:
            m = re.search(p, s, re.IGNORECASE)
            if m:
                concept = m.group(1).strip()
                definition = s.strip()
                if 1 <= len(concept.split()) <= 6:
                    concepts[concept] = definition
                break

    lines = notes_text.split("\n")
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if ":" in line:
            left, right = line.split(":", 1)
            left, right = left.strip(), right.strip()
            if left and right and len(left.split()) <= 6:
                concepts.setdefault(left, line)
        if " - " in line:
            left, right = line.split(" - ", 1)
            left, right = left.strip(), right.strip()
            if left and right and len(left.split()) <= 6:
                concepts.setdefault(left, line)
        if " – " in line:
            left, right = line.split(" – ", 1)
            left, right = left.strip(), right.strip()
            if left and right and len(left.split()) <= 6:
                concepts.setdefault(left, line)

    return concepts


def generate_flashcards_from_notes(notes_text, topic, limit=20):
    concepts = extract_concept_definitions(notes_text)
    flashcards = []

    if concepts:
        for concept, definition in concepts.items():
            flashcards.append((concept, definition))
            if len(flashcards) >= limit:
                break

    if not flashcards:
        # fallback: break notes into chunks
        sentences = split_sentences(notes_text)
        chunk_size = 2
        for i in range(0, len(sentences), chunk_size):
            chunk = " ".join(sentences[i:i + chunk_size])
            if not chunk:
                continue
            concept = f"Part {i // chunk_size + 1}"
            flashcards.append((concept, chunk))
            if len(flashcards) >= limit:
                break

    if not flashcards:
        return "❌ Not enough notes to create flashcards.", []

    md = f"### 🎴 Flashcards for **{topic}**\n\n"
    for idx, (concept, definition) in enumerate(flashcards, start=1):
        md += f"**Card {idx} – Question:** What is **{concept}**?\n\n"
        md += f"**Answer:** {definition}\n\n---\n"

    return md, flashcards


def generate_mcqs(notes_text, topic, num_q=5):
    """
    1) Try concept-based MCQs from extracted concepts
    2) Fallback: keyword blanking from sentences
    """
    notes_text = notes_text or ""
    sentences = split_sentences(notes_text)
    mcqs, answers, weak = [], [], []

    concepts = extract_concept_definitions(notes_text)
    items = list(concepts.items())
    random.shuffle(items)

    if items:
        all_concepts = [c for c, _ in items]
        for concept, definition in items:
            if len(mcqs) >= num_q:
                break
            wrong_pool = list(set(all_concepts) - {concept})
            if len(wrong_pool) < 3:
                continue
            wrongs = random.sample(wrong_pool, 3)
            options = wrongs + [concept]
            random.shuffle(options)
            correct_letter = ["A", "B", "C", "D"][options.index(concept)]
            mcqs.append({
                "question": f"What is **{concept}**?",
                "options": options,
                "answer_letter": correct_letter,
                "concept": concept
            })
            answers.append(correct_letter)
            weak.append(concept)

    if not mcqs:
        for s in sentences:
            words = extract_words(s)
            if len(words) < 5:
                continue
            key = max(words, key=len)
            wrong_pool = list(set(words) - {key})
            if len(wrong_pool) < 3:
                continue
            wrongs = random.sample(wrong_pool, 3)
            options = wrongs + [key]
            random.shuffle(options)
            correct_letter = ["A", "B", "C", "D"][options.index(key)]
            mcqs.append({
                "question": s.replace(key, "_____"),
                "options": options,
                "answer_letter": correct_letter,
                "concept": key
            })
            answers.append(correct_letter)
            weak.append(key)
            if len(mcqs) >= num_q:
                break

    return mcqs, answers, weak


def quiz_display_builder(mcq_list):
    if not mcq_list:
        return "❌ Could not generate MCQs from these notes."
    md = "### 📝 Quiz from Your Notes\n\n"
    for idx, mcq in enumerate(mcq_list, start=1):
        md += f"**Q{idx}.** {mcq['question']}\n\n"
        md += f"A) {mcq['options'][0]}\n\n"
        md += f"B) {mcq['options'][1]}\n\n"
        md += f"C) {mcq['options'][2]}\n\n"
        md += f"D) {mcq['options'][3]}\n\n"
        md += "---\n"
    return md


def quiz_generate_handler(notes_text, topic):
    topic = (topic or "").strip() or "Untitled Topic"
    if not notes_text or not notes_text.strip():
        return "❌ No notes found. Study with any technique first.", [], []

    mcqs, answers, weak = generate_mcqs(notes_text, topic)
    if not mcqs:
        return "❌ Could not generate MCQs. Add more detailed notes.", [], []

    quiz_md = quiz_display_builder(mcqs)
    return quiz_md, answers, weak


def evaluate_quiz(user_answers, correct_answers, weak_words):
    score = 0
    wrong_concepts = []

    for ua, ca, w in zip(user_answers, correct_answers, weak_words):
        if ua == ca:
            score += 1
        else:
            wrong_concepts.append(w)

    total = len(correct_answers) or 1
    percent = (score / total) * 100
    score_md = f"## 🧮 Score: **{score}/{total}** ({percent:.1f}%)\n\n"

    if wrong_concepts:
        weak_md = "### 🔎 Weak Areas\n"
        for w in wrong_concepts:
            weak_md += f"- **{w}**\n"
    else:
        weak_md = "### 🎉 No weak areas detected in this quiz."

    return percent, score_md, weak_md, wrong_concepts


def build_weakpoint_revision(notes_text, weak_words):
    if not weak_words:
        return "🎉 No weak points to revise right now."
    sentences = split_sentences(notes_text or "")
    md = "### 🔁 Quick Revision of Weak Concepts\n\n"
    for w in weak_words:
        md += f"#### 🔹 {w}\n"
        hits = [s for s in sentences if w in s][:3]
        if not hits:
            md += "(No direct sentence found in notes.)\n\n"
        else:
            for s in hits:
                md += f"- {s}\n"
            md += "\n"
    return md


# ============================================
# PART 6: Study Technique Handlers
# ============================================

def pomodoro_generate(topic, pdf, text, work_minutes, break_minutes, cycles, user):
    topic = topic.strip() or "Untitled Topic"
    combined = combine_notes(pdf, text)
    sentences = split_sentences(combined)
    if not sentences:
        sentences = [f"(No detailed notes. You can still follow Pomodoro cycles for '{topic}'.)"]

    per_chunk = ceil(len(sentences) / max(1, cycles))
    chunks = []
    for i in range(cycles):
        start = i * per_chunk
        end = start + per_chunk
        part = sentences[start:end]
        if not part:
            part = ["(Use this cycle for revision / practice.)"]
        chunks.append(" ".join(part))

    # Start with cycle 1
    current_idx = 0
    cycle_text = chunks[current_idx]

    md = f"### ⏱ Pomodoro Plan for **{topic}**\n\n"
    md += f"**Cycle {current_idx+1} / {len(chunks)}** – Focus {work_minutes} min, then break {break_minutes} min.\n\n"
    md += f"#### Notes for this cycle:\n{cycle_text[:600]}{'...' if len(cycle_text) > 600 else ''}\n\n"
    md += "_Use the next/prev buttons to move between cycles manually._"

    save_resume(user, topic, "Pomodoro", combined)
    return md, chunks, current_idx, topic, combined, "Pomodoro"


def pomodoro_next(chunks, idx, work, brk, direction):
    if not chunks:
        return "❌ No Pomodoro plan yet. Generate first.", idx
    idx = idx or 0
    if direction == "next":
        idx = min(len(chunks) - 1, idx + 1)
    elif direction == "prev":
        idx = max(0, idx - 1)

    cycle_text = chunks[idx]
    md = f"### ⏱ Pomodoro Cycle {idx+1}/{len(chunks)}\n\n"
    md += f"**Work:** {work} minutes | **Break:** {brk} minutes\n\n"
    md += f"{cycle_text[:700]}{'...' if len(cycle_text) > 700 else ''}\n\n"
    return md, idx


def flashcards_handler(topic, pdf, text, user):
    topic = topic.strip() or "Untitled Topic"
    combined = combine_notes(pdf, text)
    md, cards = generate_flashcards_from_notes(combined, topic)
    save_resume(user, topic, "Flashcards", combined)
    return md, topic, combined, "Flashcards"


def feynman_start(topic, pdf, text, user):
    topic = topic.strip() or "Untitled Topic"
    combined = combine_notes(pdf, text)
    sentences = split_sentences(combined)
    if not sentences:
        sentences = ["(Not enough notes. You can still write your own explanation.)"]
    chunk_size = 3
    chunks = []
    for i in range(0, len(sentences), chunk_size):
        chunks.append(" ".join(sentences[i:i + chunk_size]))
    idx = 0
    md = f"### 🧠 Feynman Technique – {topic}\n\n"
    md += "Read this part of your notes, then explain it in your own words below.\n\n"
    md += f"#### Part {idx+1}/{len(chunks)}:\n{chunks[idx][:800]}{'...' if len(chunks[idx]) > 800 else ''}"
    save_resume(user, topic, "Feynman", combined)
    return md, chunks, idx, topic, combined, "Feynman"


def feynman_next(chunks, idx):
    if not chunks:
        return "❌ No Feynman session yet. Generate first.", idx
    idx = idx or 0
    idx += 1
    if idx >= len(chunks):
        return "✅ You have gone through all parts of your notes using Feynman technique. Great job!", idx
    md = f"### 🧠 Feynman Technique – Next Part\n\n"
    md += f"#### Part {idx+1}/{len(chunks)}:\n{chunks[idx][:800]}{'...' if len(chunks[idx]) > 800 else ''}\n\n"
    md += "Again, try explaining this in your own words below."
    return md, idx


def active_recall_handler(topic, pdf, text, user):
    topic = topic.strip() or "Untitled Topic"
    combined = combine_notes(pdf, text)
    instructions = (
        f"### 🔁 Active Recall – {topic}\n\n"
        "1. Without looking at your notes, write everything you remember in the box.\n"
        "2. Then click **Reveal Notes** to compare.\n"
    )
    save_resume(user, topic, "Active Recall", combined)
    return instructions, topic, combined, "Active Recall"


def reveal_notes_handler(notes_text):
    if not notes_text:
        return "❌ No notes to show. Use a study tab first."
    return "### 📄 Notes Preview\n\n" + notes_text[:1500] + ("..." if len(notes_text) > 1500 else "")


def spaced_handler(topic, pdf, text, user):
    topic = topic.strip() or "Untitled Topic"
    combined = combine_notes(pdf, text)
    md = (
        f"### 🕒 Spaced Repetition Plan – {topic}\n\n"
        "- **Day 0 (Today):** Learn the topic from your notes.\n"
        "- **Day 1:** Quick review + short self-test.\n"
        "- **Day 3:** Review only weak points.\n"
        "- **Day 7:** Final review and quiz.\n"
    )
    save_resume(user, topic, "Spaced Repetition", combined)
    return md, topic, combined, "Spaced Repetition"


# ============================================
# PART 7: Helper for Home
# ============================================

def home_welcome(user):
    user = (user or "").strip()
    if user:
        return (
            f"### 👋 Welcome, **{user}**!\n\n"
            "Use the tabs above to choose how you want to study:\n"
            "- ⏱ **Pomodoro** – focus in cycles with chunked notes\n"
            "- 🎴 **Flashcards** – concept/definition cards from your notes\n"
            "- 🧠 **Feynman** – explain each part in your own words\n"
            "- 🔁 **Active Recall** – recall before seeing notes\n"
            "- 🕒 **Spaced Repetition** – plan long-term reviews\n"
            "- 📝 **Quiz & Weak Areas** – test yourself with MCQs\n"
            "- 📊 **Dashboard** – see your quiz history\n"
            "- 🔄 **Resume** – continue where you left off\n"
        )
    else:
        return (
            "### 👋 Welcome to Smart Study Hub\n\n"
            "You can use everything as a guest, but **login** to:\n"
            "- Save progress and quiz scores\n"
            "- Resume where you left off\n"
            "- See your dashboard and history\n"
        )


# ============================================
# PART 8: UI (Gradio Blocks)
# ============================================

custom_css = """
:root {
    --bg: #fdf4ff;
}

body {
    background: var(--bg);
}
.gradio-container {
    font-family: system-ui, -apple-system, BlinkMacSystemFont, "Inter", sans-serif;
}
h1, h2, h3 {
    color: #7c3aed;
}
.fade-card {
    animation: fadeIn 0.5s ease-in-out;
}
@keyframes fadeIn {
    from { opacity: 0; transform: translateY(6px); }
    to   { opacity: 1; transform: translateY(0); }
}
"""

with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as demo:
    current_user = gr.State("")
    current_topic = gr.State("")
    current_notes = gr.State("")
    current_technique = gr.State("")
    pomo_chunks_state = gr.State([])
    pomo_idx_state = gr.State(0)
    feynman_chunks_state = gr.State([])
    feynman_idx_state = gr.State(0)
    quiz_correct_state = gr.State([])
    quiz_weak_state = gr.State([])
    quiz_weak_concepts_state = gr.State([])

    gr.Markdown(
        "<h1 style='text-align:center;'>🌈 Smart Study Hub</h1>"
        "<p style='text-align:center; font-size:18px;'>Study smarter with multiple techniques, quizzes, and analytics.</p>",
        elem_classes=["fade-card"]
    )

    # ---------- Login Tab ----------
    with gr.Tab("🔐 Login / Account"):
        gr.Markdown("### Login or Sign Up")
        login_username = gr.Textbox(label="Username")
        login_password = gr.Textbox(label="Password", type="password")
        is_new_user = gr.Checkbox(label="I'm a new user (Sign up)")
        login_btn = gr.Button("Login / Sign Up", variant="primary")
        login_msg = gr.Markdown()

        login_btn.click(
            login_handler,
            inputs=[login_username, login_password, is_new_user],
            outputs=[login_msg, current_user]
        )

    # ---------- Home Tab ----------
    with gr.Tab("🏠 Home"):
        refresh_home = gr.Button("Refresh")
        home_md = gr.Markdown(elem_classes=["fade-card"])
        refresh_home.click(
            home_welcome,
            inputs=[current_user],
            outputs=[home_md]
        )

    # ---------- Pomodoro Tab ----------
    with gr.Tab("⏱ Pomodoro"):
        gr.Markdown("### Pomodoro Technique")
        p_topic = gr.Textbox(label="Topic")
        p_pdf = gr.File(label="Upload Notes (PDF)", file_types=[".pdf"])
        p_text = gr.Textbox(label="Or Paste Notes", lines=5)

        with gr.Row():
            p_work = gr.Slider(10, 50, value=25, step=5, label="Work minutes")
            p_break = gr.Slider(3, 15, value=5, step=1, label="Break minutes")
            p_cycles = gr.Slider(1, 6, value=3, step=1, label="Number of cycles")

        p_gen_btn = gr.Button("Generate Pomodoro Plan", variant="primary")
        p_prev_btn = gr.Button("◀ Previous Cycle")
        p_next_btn = gr.Button("Next Cycle ▶")
        p_md = gr.Markdown(elem_classes=["fade-card"])

        p_gen_btn.click(
            pomodoro_generate,
            inputs=[p_topic, p_pdf, p_text, p_work, p_break, p_cycles, current_user],
            outputs=[p_md, pomo_chunks_state, pomo_idx_state, current_topic, current_notes, current_technique]
        )

        p_prev_btn.click(
            pomodoro_next,
            inputs=[pomo_chunks_state, pomo_idx_state, p_work, p_break, gr.State("prev")],
            outputs=[p_md, pomo_idx_state]
        )
        p_next_btn.click(
            pomodoro_next,
            inputs=[pomo_chunks_state, pomo_idx_state, p_work, p_break, gr.State("next")],
            outputs=[p_md, pomo_idx_state]
        )

    # ---------- Flashcards Tab ----------
    with gr.Tab("🎴 Flashcards"):
        gr.Markdown("### Flashcard Study")
        f_topic = gr.Textbox(label="Topic")
        f_pdf = gr.File(label="Upload Notes (PDF)", file_types=[".pdf"])
        f_text = gr.Textbox(label="Or Paste Notes", lines=5)

        f_btn = gr.Button("Generate Flashcards", variant="primary")
        f_md = gr.Markdown(elem_classes=["fade-card"])

        f_btn.click(
            flashcards_handler,
            inputs=[f_topic, f_pdf, f_text, current_user],
            outputs=[f_md, current_topic, current_notes, current_technique]
        )

    # ---------- Feynman Tab ----------
    with gr.Tab("🧠 Feynman"):
        gr.Markdown("### Feynman Technique")
        fy_topic = gr.Textbox(label="Topic")
        fy_pdf = gr.File(label="Upload Notes (PDF)", file_types=[".pdf"])
        fy_text = gr.Textbox(label="Or Paste Notes", lines=5)

        fy_gen_btn = gr.Button("Start Feynman Session", variant="primary")
        fy_md = gr.Markdown(elem_classes=["fade-card"])
        fy_explain = gr.Textbox(label="Your explanation in simple words:", lines=5)
        fy_next_btn = gr.Button("Submit & Next Part")

        fy_gen_btn.click(
            feynman_start,
            inputs=[fy_topic, fy_pdf, fy_text, current_user],
            outputs=[fy_md, feynman_chunks_state, feynman_idx_state, current_topic, current_notes, current_technique]
        )

        fy_next_btn.click(
            feynman_next,
            inputs=[feynman_chunks_state, feynman_idx_state],
            outputs=[fy_md, feynman_idx_state]
        )

    # ---------- Active Recall Tab ----------
    with gr.Tab("🔁 Active Recall"):
        gr.Markdown("### Active Recall")
        ar_topic = gr.Textbox(label="Topic")
        ar_pdf = gr.File(label="Upload Notes (PDF)", file_types=[".pdf"])
        ar_text = gr.Textbox(label="Or Paste Notes", lines=5)

        ar_gen_btn = gr.Button("Start Active Recall", variant="primary")
        ar_inst = gr.Markdown(elem_classes=["fade-card"])
        ar_user_text = gr.Textbox(label="Write everything you remember:", lines=6)
        ar_reveal_btn = gr.Button("Reveal Notes")
        ar_notes_md = gr.Markdown()

        ar_gen_btn.click(
            active_recall_handler,
            inputs=[ar_topic, ar_pdf, ar_text, current_user],
            outputs=[ar_inst, current_topic, current_notes, current_technique]
        )

        ar_reveal_btn.click(
            reveal_notes_handler,
            inputs=[current_notes],
            outputs=[ar_notes_md]
        )

    # ---------- Spaced Repetition Tab ----------
    with gr.Tab("🕒 Spaced Repetition"):
        gr.Markdown("### Spaced Repetition")
        sp_topic = gr.Textbox(label="Topic")
        sp_pdf = gr.File(label="Upload Notes (PDF)", file_types=[".pdf"])
        sp_text = gr.Textbox(label="Or Paste Notes", lines=5)

        sp_btn = gr.Button("Generate Spaced Plan", variant="primary")
        sp_md = gr.Markdown(elem_classes=["fade-card"])

        sp_btn.click(
            spaced_handler,
            inputs=[sp_topic, sp_pdf, sp_text, current_user],
            outputs=[sp_md, current_topic, current_notes, current_technique]
        )

    # ---------- Quiz Tab ----------
    with gr.Tab("📝 Quiz & Weak Areas"):
        gr.Markdown("### MCQ Quiz from Your Notes")

        q_gen_btn = gr.Button("Generate Quiz from Last Notes", variant="primary")
        q_md = gr.Markdown(elem_classes=["fade-card"])

        ans1 = gr.Dropdown(["", "A", "B", "C", "D"], label="Answer for Q1")
        ans2 = gr.Dropdown(["", "A", "B", "C", "D"], label="Answer for Q2")
        ans3 = gr.Dropdown(["", "A", "B", "C", "D"], label="Answer for Q3")
        ans4 = gr.Dropdown(["", "A", "B", "C", "D"], label="Answer for Q4")
        ans5 = gr.Dropdown(["", "A", "B", "C", "D"], label="Answer for Q5")

        submit_q_btn = gr.Button("Submit Quiz", variant="primary")
        score_out = gr.Markdown()
        weak_out = gr.Markdown()
        revision_out = gr.Markdown()

        study_weak_btn = gr.Button("Study Weak Points Again")
        back_home_btn = gr.Button("Back to Home Message")

        q_gen_btn.click(
            quiz_generate_handler,
            inputs=[current_notes, current_topic],
            outputs=[q_md, quiz_correct_state, quiz_weak_state]
        )

        def quiz_submit_wrapper(a1, a2, a3, a4, a5,
                                correct, weak, user, topic, technique, notes):
            correct = correct or []
            weak = weak or []
            user_answers = [a1, a2, a3, a4, a5][:len(correct)]
            if not correct:
                return "❌ Generate a quiz first.", "ℹ️ No weak areas yet.", []
            percent, score_md, weak_md, wrong_concepts = evaluate_quiz(
                user_answers, correct, weak
            )
            save_progress(user, topic or "Untitled Topic", technique or "Quiz", percent)
            save_resume(user, topic or "Untitled Topic", technique or "Quiz", notes or "")
            return score_md, weak_md, wrong_concepts

        submit_q_btn.click(
            quiz_submit_wrapper,
            inputs=[
                ans1, ans2, ans3, ans4, ans5,
                quiz_correct_state, quiz_weak_state,
                current_user, current_topic, current_technique, current_notes
            ],
            outputs=[score_out, weak_out, quiz_weak_concepts_state]
        )

        study_weak_btn.click(
            build_weakpoint_revision,
            inputs=[current_notes, quiz_weak_concepts_state],
            outputs=[revision_out]
        )

        back_home_btn.click(
            lambda: "🏠 Go to the **Home** tab and choose your next study mode.",
            inputs=[],
            outputs=[revision_out]
        )

    # ---------- Resume Tab ----------
    with gr.Tab("🔄 Resume"):
        gr.Markdown("### Continue Where You Left Off")
        res_btn = gr.Button("Load Last Session", variant="primary")
        res_md = gr.Markdown(elem_classes=["fade-card"])
        res_btn.click(
            load_last_resume,
            inputs=[current_user],
            outputs=[res_md, current_topic, current_notes, current_technique]
        )

    # ---------- Dashboard Tab ----------
    with gr.Tab("📊 Dashboard"):
        gr.Markdown("### Your Quiz Progress")
        dash_btn = gr.Button("Refresh Dashboard", variant="primary")
        dash_md = gr.Markdown(elem_classes=["fade-card"])
        dash_btn.click(
            render_dashboard,
            inputs=[current_user],
            outputs=[dash_md]
        )

    # ---------- Help Tab ----------
    with gr.Tab("❓ Help / About"):
        gr.Markdown(
            """
            ### ℹ️ About Smart Study Hub

            - Built in **Python** using **Gradio**.
            - Allows multiple study techniques:
              - Pomodoro with note chunks
              - Flashcards from concepts/definitions
              - Feynman explanation in parts
              - Active Recall
              - Spaced Repetition
            - Generates MCQ quiz from your own notes.
            - Tracks weak areas and suggests revision.
            - Login system + Resume + Dashboard using CSV storage.

            **Suggested flow:**
            1. Log in on the **Login** tab.
            2. Use **Pomodoro / Flashcards / Feynman / etc.** to study.
            3. Go to **Quiz & Weak Areas** and test yourself.
            4. Use **Dashboard** to track progress.
            5. Use **Resume** to pick up where you left off.
            """
        )

# ============================================
# PART 9: Launch
# ============================================

if __name__ == "__main__":
    demo.launch()


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
